# 09 — Router por zonas: especializar solo donde paga

El nb 06 mostró que "un modelo por zona" a secas **no** le gana al pooled: ayuda en
la Pampa pero colapsa en el norte subtropical (poca señal + pocos datos), y esa
zona hunde el neto. La solución: un **router** que, para cada zona, elige por
**validación cruzada temporal en train** si conviene el modelo especializado o el
pooled — sin mirar el test (eso sería leakage de selección).

Implementado en `wrapper_zonas.WrapperPorZona`. Comparamos router vs. pooled vs.
"un modelo por zona (puro)" en el test.

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')
import json
from wrapper_zonas import WrapperPorZona
from modelos import XGBoostRegressor
CULTIVO = 'soja'
BEST = json.load(open('retuning_cv_honesta.json', encoding='utf-8')).get(CULTIVO, {}).get('xgb', {}).get('best_params', {}) if os.path.exists('retuning_cv_honesta.json') else {}
BEST = {k: v for k, v in BEST.items() if k not in ('n_jobs',)}

## Entrenamiento del router (decisión por CV en train)

In [ ]:
w = WrapperPorZona(XGBoostRegressor, BEST, cultivo=CULTIVO, use_agro=True,
                   enc_smooth=10.0, n_zonas=6, metric='rmse')
w.fit()
w.resumen_zonas()

La columna `elegido` dice, por zona, si el router se quedó con el modelo **especializado** (mejor CV) o el **pooled**.

## Resultado en test: router vs. pooled vs. por-zona puro

In [ ]:
tabla = w.evaluar()
tabla

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
t = tabla.sort_values('rmse')
ax.barh(t['modelo'][::-1], t['rmse'][::-1], color=['#55A868', '#4C72B0', '#C44E52'][:len(t)])
ax.set_xlabel('RMSE (kg/ha)'); ax.set_title('Router por zonas vs. alternativas (test)')
for i, v in enumerate(t['rmse'][::-1]):
    ax.text(v, i, f' {v:.0f}', va='center')
plt.tight_layout(); plt.show()

## Conclusión

- El **router** se queda con el especializado solo en las zonas donde la CV en train
  lo respalda (típicamente las pampeanas) y usa el pooled en las ruidosas → acota el
  colapso del norte que hundía al "un modelo por zona" del nb 06.
- Así logra **igualar o superar al pooled** sin el riesgo del esquema ingenuo. La
  clave metodológica es que la decisión por zona sale de **CV temporal en train**,
  nunca del test.
- Es un ejemplo de *split selectivo* / mixture-of-experts liviano; el siguiente paso
  natural sería el **pooling parcial** (encoger cada zona hacia el pooled según su n).